# 03 — Enrich Geography and Group Campuses

## Research purpose

Add geographic attributes to the hyperscaler candidates, remove exact source duplicates and group facilities into campuses.


## Imports

Load the libraries used for country lookup, time-zone assignment and campus grouping.


In [1]:
from collections import defaultdict
from datetime import datetime, timezone
import math
from pathlib import Path

import pandas as pd
import pycountry
import pytz
import reverse_geocoder as rg
from timezonefinder import TimezoneFinder

## Project location and paths

Identify the project directory and define the folders for the manual country mapping, candidate records and enriched facility tables.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "notebooks").is_dir() or not (PROJECT_ROOT / "data").is_dir():
    raise RuntimeError("Launch Jupyter from the folder containing data/ and notebooks/, or from its notebooks/ directory.")

DATA_MANUAL = PROJECT_ROOT / "data" / "manual"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

## Geographic settings

Use 2 June 2026 at 12:00 UTC as the reference instant for time-zone offsets. Set the campus grouping distance to 1 km.


In [3]:
GEO_REFERENCE_TIMESTAMP_UTC = datetime(2026, 6, 2, 12, 0, tzinfo=timezone.utc)
CAMPUS_GROUP_DISTANCE_M = 1000.0

## Load and validate candidates

Load the hyperscaler candidates and convert their coordinates to numeric values. Check for missing coordinates before assigning geographic attributes or calculating distances.


In [4]:
df = pd.read_csv(DATA_INTERIM / "datacenters_hyperscaler_candidates.csv")

for column in ["latitude", "longitude"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

if df[["latitude", "longitude"]].isna().any().any():
    raise ValueError("Candidate table contains invalid coordinates")

print(f"Candidates: {len(df):,}")

Candidates: 511


## Assign geographic attributes

Assign countries and time zones from the facility coordinates and region labels from the manual country mapping. Calculate UTC offsets at the fixed reference instant and save the enriched records. If a known country lacks a region mapping, write the affected records to a review table and stop.


In [5]:
# Resolve an ISO alpha-2 code to its country name.
def country_name(alpha2):
    if not alpha2:
        return None
    country = pycountry.countries.get(alpha_2=alpha2)
    return country.name if country else None

# Resolve an ISO alpha-2 code to the exported alpha-3 country code.
def alpha3(alpha2):
    if not alpha2:
        return None
    country = pycountry.countries.get(alpha_2=alpha2)
    return country.alpha_3 if country else None

region_lookup = pd.read_csv(DATA_MANUAL / "country_region_mapping.csv").set_index("country_code_alpha2")[["world_region", "sub_region"]].to_dict("index")

# Read region labels from the researcher-authored country mapping.
def region(alpha2):
    values = region_lookup.get(alpha2)
    return (values["world_region"], values["sub_region"]) if values else ("unknown", "unknown")

tf = TimezoneFinder()
coordinates = list(zip(df["latitude"].astype(float), df["longitude"].astype(float)))
rg_results = rg.search(coordinates, mode=1)

df["country_code_alpha2"] = [result.get("cc") for result in rg_results]
df["country"] = df["country_code_alpha2"].apply(country_name)
df["country_code"] = df["country_code_alpha2"].apply(alpha3)
df[["world_region", "sub_region"]] = pd.DataFrame(df["country_code_alpha2"].apply(region).tolist(), index=df.index)
df["timezone"] = df.apply(lambda row: tf.timezone_at(lat=row["latitude"], lng=row["longitude"]), axis=1)


# Calculate the timezone offset at the declared fixed reference instant.
def utc_offset(timezone_name):
    if not timezone_name:
        return None
    zone = pytz.timezone(timezone_name)
    return GEO_REFERENCE_TIMESTAMP_UTC.astimezone(zone).utcoffset().total_seconds() / 3600

df["utc_offset"] = df["timezone"].apply(utc_offset)

unmapped = df[df["country_code_alpha2"].notna() & df["world_region"].eq("unknown")]
if not unmapped.empty:
    unmapped.to_csv(DATA_INTERIM / "datacenters_unmapped_country_review.csv", index=False)
    raise ValueError(f"{len(unmapped)} records have known country codes without region mappings")
df.to_csv(DATA_PROCESSED / "datacenters_hyperscalers_enriched.csv", index=False)
df[["facility_name_raw", "operator_group", "country", "country_code", "timezone"]].head()

Loading formatted geocoded file...


,facility_name_raw,operator_group,country,country_code,timezone
0,Google Fiber,Google,United States,USA,America/New_York
1,Amazon SIN4,Amazon,Singapore,SGP,Asia/Singapore
2,Amazon SIN2,Amazon,Malaysia,MYS,Asia/Singapore
3,Google Singapore DC,Google,Malaysia,MYS,Asia/Singapore
4,Amazon SIN3,Amazon,Singapore,SGP,Asia/Singapore


## Duplicate and campus grouping methods

Define functions to identify exact source duplicates and group nearby facilities belonging to the same operator. Campus grouping is transitive, so facilities connected through neighbours can share a campus even when the endpoints are more than 1 km apart. The grouping preserves individual facility rows.


In [6]:
# Measure great-circle separation in metres for the campus proximity rule.
def haversine_m(lat1, lon1, lat2, lon2):
    radius = 6371008.8
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * radius * math.asin(math.sqrt(a))


class UnionFind:
    # Start each record in its own connected component.
    def __init__(self, values):
        self.parent = {value: value for value in values}

    # Find the component representative while shortening the parent chain.
    def find(self, value):
        while self.parent[value] != value:
            self.parent[value] = self.parent[self.parent[value]]
            value = self.parent[value]
        return value

    # Join the connected components of two nearby same-operator records.
    def union(self, left, right):
        root_left = self.find(left)
        root_right = self.find(right)
        if root_left != root_right:
            self.parent[root_right] = root_left

    # Collect the record indices belonging to each connected component.
    def groups(self):
        grouped = defaultdict(list)
        for value in self.parent:
            grouped[self.find(value)].append(value)
        return list(grouped.values())


# Build transitive proximity groups without merging records from different operators.
def group_nearby_same_operator(df, distance_m: float):
    """Return transitive same-operator proximity groups without collapsing rows."""
    ids = list(df.index)
    uf = UnionFind(ids)
    for i, left_id in enumerate(ids):
        left = df.loc[left_id]
        for right_id in ids[i + 1 :]:
            right = df.loc[right_id]
            if left["operator_group"] != right["operator_group"]:
                continue
            distance = haversine_m(
                float(left["latitude"]),
                float(left["longitude"]),
                float(right["latitude"]),
                float(right["longitude"]),
            )
            if distance <= distance_m:
                uf.union(left_id, right_id)
    return uf.groups()


# Group exact source keys and leave records without a key as singletons.
def exact_duplicate_groups(df, key_column: str = "osm_ref"):
    """Return groups that share an exact source key, plus singleton groups."""
    grouped = defaultdict(list)
    for idx, row in df.iterrows():
        key = row.get(key_column)
        if key is None or key != key:
            key = f"row:{idx}"
        grouped[str(key)].append(idx)
    return list(grouped.values())

## Prepare polygon areas

Convert polygon areas to numeric values so they can be used to select a representative from each group of exact duplicates.


In [7]:
if "polygon_area_m2" in df.columns:
    df["polygon_area_m2"] = pd.to_numeric(df["polygon_area_m2"], errors="coerce")

## Select duplicate representatives

Select one record per OSM reference, prioritising geometry type and then the largest polygon area. Preserve the source identifiers and collect duplicate groups for review, while keeping distinct buildings as separate records.


In [8]:
exact_groups = exact_duplicate_groups(df, key_column="osm_ref")

priority = {
    "building_footprint": 0,
    "point": 1,
    "site_or_landuse_polygon": 2,
    "unclassified_polygon": 3,
}

def choose_representative(group_df):
    temp = group_df.copy()
    temp["geometry_priority"] = temp["geometry_kind"].map(priority).fillna(9)
    temp["area_sort"] = pd.to_numeric(temp.get("polygon_area_m2"), errors="coerce").fillna(0)
    return temp.sort_values(["geometry_priority", "area_sort"], ascending=[True, False]).iloc[0]

records = []
review_rows = []
for number, group_indices in enumerate(exact_groups, start=1):
    group_df = df.loc[group_indices].copy()
    representative = choose_representative(group_df).copy()
    source_ids = sorted(group_df["source_id"].dropna().astype(str).unique())
    representative["duplicate_group_id"] = f"DUP-{number:05d}"
    representative["duplicate_status"] = "singleton" if len(group_df) == 1 else "exact_duplicate_removed"
    representative["deduplication_method"] = "same_osm_ref_exact"
    representative["source_ids"] = "|".join(source_ids)
    records.append(representative)
    if len(group_df) > 1:
        group_df["duplicate_group_id"] = representative["duplicate_group_id"]
        group_df["duplicate_status"] = "exact_duplicate_removed"
        review_rows.append(group_df)

dedup = pd.DataFrame(records).reset_index(drop=True)

## Assign campuses and capacity roles

Assign each facility a campus identifier and group size using the same-operator distance rule. Mark site boundaries as context-only when building records already represent the campus and assign the remaining geometry roles. Save the facility table and the duplicate and campus review tables.


In [9]:
# Non-collapsing campus groups: useful for later aggregation and review, but rows stay facility-level.
campus_groups = group_nearby_same_operator(dedup, distance_m=CAMPUS_GROUP_DISTANCE_M)

dedup["campus_group_id"] = pd.NA
dedup["campus_group_size"] = 1
dedup["capacity_record_role"] = "facility_candidate"

for number, group_indices in enumerate(campus_groups, start=1):
    campus_id = f"CAMPUS-{number:05d}"
    group_df = dedup.loc[group_indices].copy()
    has_building = (group_df["geometry_kind"] == "building_footprint").any()
    dedup.loc[group_indices, "campus_group_id"] = campus_id
    dedup.loc[group_indices, "campus_group_size"] = len(group_df)
    site_mask = dedup.index.isin(group_indices) & (dedup["geometry_kind"] == "site_or_landuse_polygon")
    if has_building:
        dedup.loc[site_mask, "capacity_record_role"] = "site_context_only"
    else:
        dedup.loc[site_mask, "capacity_record_role"] = "site_capacity_proxy"

point_mask = dedup["geometry_kind"] == "point"
dedup.loc[point_mask, "capacity_record_role"] = "point_facility_proxy"
review_mask = dedup["geometry_kind"] == "unclassified_polygon"
dedup.loc[review_mask, "capacity_record_role"] = "geometry_review"

review = pd.concat(review_rows, ignore_index=True) if review_rows else pd.DataFrame(columns=df.columns)
campus_review = dedup[dedup["campus_group_size"].gt(1)].copy()

dedup.to_csv(DATA_PROCESSED / "datacenters_hyperscalers_deduplicated.csv", index=False)
review.to_csv(DATA_INTERIM / "datacenters_dedup_candidates.csv", index=False)
campus_review.to_csv(DATA_INTERIM / "datacenters_campus_groups.csv", index=False)

print(f"Input rows: {len(df):,}")
print(f"Deduplicated rows: {len(dedup):,}")
print(f"True duplicate review rows: {len(review):,}")
print(f"Rows in multi-record campus groups: {len(campus_review):,}")
print(dedup["capacity_record_role"].value_counts(dropna=False))
dedup[["duplicate_group_id", "campus_group_id", "capacity_record_role", "facility_name_raw", "operator_group", "osm_ref"]].head()

Input rows: 511
Deduplicated rows: 511
True duplicate review rows: 0
Rows in multi-record campus groups: 437
capacity_record_role
facility_candidate      453
point_facility_proxy     20
site_context_only        19
site_capacity_proxy      15
geometry_review           4
Name: count, dtype: int64


,duplicate_group_id,campus_group_id,capacity_record_role,facility_name_raw,operator_group,osm_ref
0,DUP-00001,CAMPUS-00001,point_facility_proxy,Google Fiber,Google,node/4879481820
1,DUP-00002,CAMPUS-00002,point_facility_proxy,Amazon SIN4,Amazon,node/8208591051
2,DUP-00003,CAMPUS-00003,point_facility_proxy,Amazon SIN2,Amazon,node/8208591054
3,DUP-00004,CAMPUS-00004,point_facility_proxy,Google Singapore DC,Google,node/8208591112
4,DUP-00005,CAMPUS-00005,point_facility_proxy,Amazon SIN3,Amazon,node/8208598531


## Validate campus operators

Check that every campus group contains facilities from only one operator.


In [10]:
assert not dedup.groupby("campus_group_id")["operator_group"].nunique().gt(1).any()